In [1]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
# PR10: Dimensionality Reduction using PCA - Heart Disease Dataset
file_path = 'heart.csv'
df = pd.read_csv(file_path)
print("Dataset loaded successfully! Shape:", df.shape)
display(df.head(3))

# Identify target column
target_col = 'target' if 'target' in df.columns else df.columns[-1]
y = df[target_col].values
X = df.drop(columns=[target_col])

# Encode categorical features if present
for col in X.select_dtypes(include=['object']).columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Split first, then scale (scaler fitted on train only to avoid leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

Dataset loaded successfully! Shape: (920, 16)


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1


C:\Users\omkar\AppData\Local\Temp\ipykernel_21104\1634429000.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=['object']).columns:


In [3]:
# Scree plot: individual and cumulative explained variance
pca_full = PCA().fit(X_train_s)
var = pca_full.explained_variance_ratio_ * 100
cum_var = np.cumsum(var)
pcs = np.arange(1, len(var) + 1)

plt.figure(figsize=(9, 5))
plt.plot(pcs, var, 'o-', label='Individual variance (%)')
plt.plot(pcs, cum_var, 's--', label='Cumulative variance (%)')
plt.axhline(95, color='gray', ls=':', label='95% threshold')
plt.title('Scree Plot / Explained Variance')
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained (%)')
plt.xticks(pcs)
plt.legend(); plt.grid(True); plt.show()

n_95 = np.argmax(cum_var >= 95) + 1
print(f"Components needed for 95% variance: {n_95}")

ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# 2D PCA scatter plot
pca2 = PCA(n_components=2)
X_pca2 = pca2.fit_transform(X_train_s)

pca_df = pd.DataFrame(X_pca2, columns=['Principal Component 1', 'Principal Component 2'])
pca_df['Target'] = y_train

plt.figure(figsize=(8, 6))
sns.scatterplot(x='Principal Component 1', y='Principal Component 2',
                hue='Target', data=pca_df, palette='Set1', alpha=0.8)
plt.title('PCA: 2D Projection of Heart Disease Dataset')
plt.xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.grid(True); plt.show()

In [ ]:
# Feature count and performance comparison
def evaluate(Xtr, Xte):
    start = time.perf_counter()
    model = LogisticRegression(max_iter=1000).fit(Xtr, y_train)
    acc = accuracy_score(y_test, model.predict(Xte))
    return acc, time.perf_counter() - start

results = []

# Before PCA
acc, t = evaluate(X_train_s, X_test_s)
results.append(["Before PCA", X_train_s.shape[1], acc, t])

# After PCA: 2 components, and components covering 95% variance
for name, n in [("PCA (2 components)", 2), (f"PCA (95% variance, {n_95} comps)", n_95)]:
    p = PCA(n_components=n)
    start = time.perf_counter()
    Xtr_p = p.fit_transform(X_train_s)
    Xte_p = p.transform(X_test_s)
    pca_time = time.perf_counter() - start
    acc, t = evaluate(Xtr_p, Xte_p)
    results.append([name, n, acc, t + pca_time])  # includes PCA time

res_df = pd.DataFrame(results, columns=["Model", "No. of Features", "Accuracy", "Execution Time (s)"])
res_df["Accuracy"] = res_df["Accuracy"].round(4)
res_df["Execution Time (s)"] = res_df["Execution Time (s)"].round(5)

print("--- Feature Count & Performance Comparison ---")
display(res_df)